# MCX Gold: cross-contract relative value (Summit Syndicate)
Run the cells top to bottom (Runtime > Run all). In cell 1, select all your Bhavcopy CSV files at once.

In [ ]:
import os
os.makedirs("data", exist_ok=True)
try:
    from google.colab import files
    up = files.upload()                      # choose all Bhavcopy CSVs
    for name, content in up.items():
        open(f"data/{name}", "wb").write(content)
except ImportError:
    print("Not on Colab: put CSVs in ./data")

In [ ]:
import glob
import pandas as pd
import matplotlib.pyplot as plt

# ---- CONFIG: verify against the MCX gold contract specs page ----
QUOTE_BASE_G = {"GOLDM": 10, "GOLDTEN": 10, "GOLDGUINEA": 8, "GOLDPETAL": 1}   # grams the quoted price refers to
PURITY = {"GOLDM": 995, "GOLDTEN": 999, "GOLDGUINEA": 999, "GOLDPETAL": 999}
BASIS = 995

def load(folder="data"):
    files = glob.glob(f"{folder}/*.csv")
    assert files, "No CSV files in data/"
    df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    df.columns = [c.strip() for c in df.columns]
    df["Symbol"] = df["Symbol"].astype(str).str.strip().str.upper()      # symbols can be space-padded
    df = df[df["Symbol"].isin(QUOTE_BASE_G)].copy()
    d = pd.to_datetime(df["Date"], format="%m/%d/%Y", errors="coerce")   # MM/DD/YYYY first, then DD/MM/YYYY
    df["Date"] = d.fillna(pd.to_datetime(df["Date"], dayfirst=True, errors="coerce"))
    df["ExpiryDate"] = pd.to_datetime(df["ExpiryDate"].astype(str).str.strip(), format="%d%b%Y", errors="coerce")
    for col in ["Open", "High", "Low", "Close", "Volume", "OpenInterest"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.dropna(subset=["Date", "ExpiryDate", "Close"])
    df = df[df["Date"] <= df["ExpiryDate"]]
    return df.drop_duplicates(["Date", "Symbol", "ExpiryDate"]).sort_values(["Date", "Symbol"])

def normalize(df):
    df = df.copy()
    df["rs_per_g"] = df["Close"] / df["Symbol"].map(QUOTE_BASE_G) * BASIS / df["Symbol"].map(PURITY)
    df["contract"] = df["Symbol"] + " " + df["ExpiryDate"].dt.strftime("%b%Y")   # keyed by expiry, not near-month
    df["exp_month"] = df["ExpiryDate"].dt.to_period("M")
    return df

def spread(df, a, b, min_obs=3, k=2.0):
    x = df[df["Symbol"] == a].set_index(["Date", "exp_month"])["rs_per_g"]
    y = df[df["Symbol"] == b].set_index(["Date", "exp_month"])["rs_per_g"]
    s = (x - y).dropna().rename("spread").reset_index().sort_values("Date")
    out = []
    for m, g in s.groupby("exp_month"):
        g = g.copy(); past = g["spread"].shift(1)                  # band uses PAST days only: no look-ahead
        g["mean"] = past.expanding(min_obs).mean(); g["std"] = past.expanding(min_obs).std()
        g["z"] = (g["spread"] - g["mean"]) / g["std"]; g["flag"] = g["z"].abs() > k
        out.append(g)
    return pd.concat(out) if out else s

df = normalize(load("data"))
print(f"{df['Date'].min().date()} to {df['Date'].max().date()} | {df['Date'].nunique()} trading days | {df['contract'].nunique()} contracts")

## Table for the Market slide: volume and open interest from real data

In [ ]:
last = df["Date"].max()
tbl = df.groupby("Symbol").agg(avg_daily_volume=("Volume", "mean"), total_volume=("Volume", "sum")).round(0)
tbl["open_interest_last_day"] = df[df["Date"] == last].groupby("Symbol")["OpenInterest"].sum()
tbl

## Chart 1: all contracts on one basis (Rs per gram, 995 purity)

In [ ]:
wide = df.pivot_table(index="Date", columns="contract", values="rs_per_g")
ax = wide.plot(figsize=(11, 5), marker="o"); ax.set_ylabel("Rs per gram (995 basis)"); ax.set_title("Four contracts on one basis"); plt.grid(alpha=.3); plt.show()

## Chart 2: spread vs its normal range (change A, B, K)

In [ ]:
A, B, K = "GOLDM", "GOLDPETAL", 2.0
sp = spread(df, A, B, k=K)
if sp.empty:
    print("No same-expiry-month overlap between these two contracts in your data.")
else:
    for m, g in sp.groupby("exp_month"):
        fig, ax = plt.subplots(figsize=(11, 4))
        ax.plot(g["Date"], g["spread"], marker="o", label=f"{A} - {B}")
        ax.fill_between(g["Date"], g["mean"] - K*g["std"], g["mean"] + K*g["std"], alpha=.2, label=f"normal range (+/-{K} sd)")
        ax.scatter(g[g["flag"]]["Date"], g[g["flag"]]["spread"], color="red", zorder=5, label="flag")
        ax.set_title(f"Spread, expiry month {m}"); ax.set_ylabel("Rs per gram"); ax.legend(); ax.grid(alpha=.3); plt.show()
    hits = sp[sp["flag"]]
    print("QUIET: no meaningful signal. (No edge is a valid result.)" if hits.empty else f"{len(hits)} day(s) outside the normal range:")
    if not hits.empty: display(hits[["Date", "exp_month", "spread", "z"]])
print("Costs and slippage are NOT applied: a flag is a candidate, not a trade.")